# MV-VTON — пилот на пальто Silver.Vy

Ноутбук готовит изображения для тестового загрузчика MV-VTON и запускает одну генерацию. Используются один фронтальный снимок человека, полный фронт пальто, крупный фронтальный ракурс и боковой ракурс. Код модели использует фронтальный и второй ракурс как замену заднего; боковой ракурс не равен заднему, поэтому такой запуск — эксперимент, не проверка штатного режима.

В Colab выбери **Среда выполнения → Сменить среду выполнения → T4 GPU**. После установки загрузи 4 файла: `person.jpg`, `coat-front.png`, `coat-detail.png`, `coat-side.png`. Для теста можно взять уже подготовленного синтетического человека и три изображения пальто из каталога.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU не включен. Выбери T4 GPU в настройках среды выполнения.')
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
!git clone -q https://github.com/hywang2002/MV-VTON.git /content/MV-VTON
%cd /content/MV-VTON
!pip -q install pytorch-lightning omegaconf einops opencv-python-headless transformers controlnet-aux gdown
!pip -q install git+https://github.com/openai/CLIP.git

In [ ]:
from google.colab import files
uploaded = files.upload()
required = {'person.jpg', 'coat-front.png', 'coat-detail.png', 'coat-side.png'}
missing = required - set(uploaded)
if missing:
    raise ValueError(f'Не загружены файлы: {sorted(missing)}')

In [ ]:
import os, shutil, json
import numpy as np
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
from controlnet_aux import OpenposeDetector
import torch.nn.functional as F

ROOT = Path('/content/MV-VTON')
DATA = Path('/content/MVG/test')
PERSON_ID = '00001_1'
CLOTH_ID = '00001_1'
NAMES = ['1.png', '2.png', '5.png']
model_id = 'mattmdjaga/segformer_b2_clothes'
processor = SegformerImageProcessor.from_pretrained(model_id)
parser = SegformerForSemanticSegmentation.from_pretrained(model_id).to('cuda').eval()
pose_detector = OpenposeDetector.from_pretrained('lllyasviel/Annotators')

def open_rgb(path):
    image = ImageOps.exif_transpose(Image.open(path))
    if image.mode == 'P' and 'transparency' in image.info:
        image = image.convert('RGBA')
    if 'A' in image.getbands():
        background = Image.new('RGBA', image.size, (128, 128, 128, 255))
        image = Image.alpha_composite(background, image)
    return image.convert('RGB')

def parse_mask(image, class_ids):
    inputs = processor(images=image, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        logits = parser(**inputs).logits
        logits = F.interpolate(logits, size=(image.height, image.width), mode='bilinear', align_corners=False)[0]
        probabilities = logits.softmax(dim=0)
        confidence, labels = probabilities.max(dim=0)
    selected = torch.zeros_like(labels, dtype=torch.bool)
    for class_id in class_ids:
        selected |= labels == class_id
    return Image.fromarray(((selected & (confidence >= 0.42)).byte().cpu().numpy() * 255).astype('uint8'))

def square(image, fill):
    side = max(image.size)
    canvas = Image.new(image.mode, (side, side), fill)
    canvas.paste(image, ((side-image.width)//2, (side-image.height)//2))
    return canvas.resize((512, 512), Image.Resampling.LANCZOS)

def save_masked(image, mask, path):
    image = square(image, (128, 128, 128))
    mask = square(mask, 0)
    image.save(path)
    mask.save(path.parent.parent / 'cloth-mask' / path.parent.name / path.name)
    return image, mask

for folder in ['image-wo-bg', 'cloth', 'cloth-mask', 'inpaint_mask', 'skeletons', 'warp_feat_unpair']:
    (DATA / folder).mkdir(parents=True, exist_ok=True)
person_dir = DATA / 'image-wo-bg' / PERSON_ID
cloth_dir = DATA / 'cloth' / CLOTH_ID
cloth_mask_dir = DATA / 'cloth-mask' / CLOTH_ID
for folder in [person_dir, cloth_dir, cloth_mask_dir, DATA / 'inpaint_mask' / PERSON_ID, DATA / 'skeletons' / PERSON_ID, DATA / 'skeletons' / CLOTH_ID]:
    folder.mkdir(parents=True, exist_ok=True)

person = open_rgb('/content/person.jpg')
person_mask = parse_mask(person, {4})
person_512 = square(person, (128, 128, 128))
person_mask_512 = square(person_mask, 0)
agnostic = person_512.copy()
agnostic.paste((128, 128, 128), mask=person_mask_512)
agnostic.save(DATA / 'warp_feat_unpair' / f'{PERSON_ID}.jpg')
person_512.save(person_dir / '2.png')
person_512.save(person_dir / '1.png')
person_512.save(person_dir / '5.png')
person_mask_512.save(DATA / 'inpaint_mask' / PERSON_ID / '2.png')
pose_person = pose_detector(person_512)
pose_person.save(DATA / 'skeletons' / PERSON_ID / '2.png')
pose_person.save(DATA / 'skeletons' / PERSON_ID / '1.png')
pose_person.save(DATA / 'skeletons' / PERSON_ID / '5.png')

views = [('coat-front.png', '1.png'), ('coat-detail.png', '2.png'), ('coat-side.png', '5.png')]
for filename, output_name in views:
    garment = open_rgb('/content/' + filename)
    garment_mask = parse_mask(garment, {4, 7})
    garment_512 = square(garment, (128, 128, 128))
    garment_mask_512 = square(garment_mask, 0)
    garment_512.save(cloth_dir / output_name)
    garment_mask_512.save(cloth_mask_dir / output_name)
    pose = pose_detector(garment_512)
    pose.save(DATA / 'skeletons' / CLOTH_ID / output_name)
    preview = garment_512.convert('RGBA')
    tint = Image.new('RGBA', preview.size, (30, 210, 140, 0))
    tint.putalpha(garment_mask_512.point(lambda value: value * 110 // 255))
    Image.alpha_composite(preview, tint).convert('RGB').save('/content/' + output_name.replace('.png', '-mask-review.jpg'))

(DATA.parent / 'unpaired.txt').write_text(f'{PERSON_ID.split("_")[0]} {CLOTH_ID.split("_")[0]}\n')
print('Подготовка завершена. Перед генерацией проверь mask-review.jpg и person-agnostic.')

## Проверка масок

Перед запуском посмотри все три `*-mask-review.jpg`. Сегментатор часто путает длинное пальто с платьем, а открытую часть пальто — с одеждой под ним. Если маска включает лицо, кисти, брюки или пропускает рукава/полы, генерацию пока не запускай: сначала исправь маску вручную. Автоматическая маска — черновик, не подтверждение правильной сегментации.

In [ ]:
from IPython.display import display
for name in ['1-mask-review.jpg', '2-mask-review.jpg', '5-mask-review.jpg']:
    display(Image.open('/content/' + name))
display(Image.open(DATA / 'warp_feat_unpair' / f'{PERSON_ID}.jpg'))

In [ ]:
!cp ldm/data/cp_dataset_mv_unpaired.py ldm/data/cp_dataset.py
!mkdir -p models/vgg checkpoints
!gdown -q 1rvow8jStPt8t2prDcSRlnf8yzXhrYeGo -O models/vgg/vgg.pth
!gdown -q 1J91PoT8A9yqHWNxkgRe6ZCnDEhN-H9O6 -O checkpoints/mvg.ckpt
!python test.py --gpu_id 0 --ddim_steps 50 --outdir /content/mv-vton-results --config configs/viton512.yaml --dataroot /content/MVG --ckpt checkpoints/mvg.ckpt --n_samples 1 --seed 23 --scale 1 --H 512 --W 384

## Как оценивать результат

Сначала оцени только форму и сохранение пальто. MV-VTON не принимает размер пользователя и не обещает точную посадку. Не сравнивай размеры по тексту: для размерного слоя сначала потребуются реальные мерки пальто по каждому размеру и мерки тела, затем отдельная геометрическая коррекция. Если установка падает из-за совместимости старого репозитория с текущим Colab, сохрани полный traceback — не переустанавливай случайные версии CUDA и PyTorch поверх работающей среды.